In [60]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.agents import create_agent
from langchain.tools import tool
from dotenv import load_dotenv

from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings

embedder = NVIDIAEmbeddings(model="NV-Embed-QA")


load_dotenv()

True

In [61]:
# Step 1 : Load the document from source ( PDF, TEXT, WEB etc)
loader = PyPDFLoader('../data/medical_report.pdf')
docs = loader.load()
len(docs)

9

In [62]:
# Step 2: Split the Document which is loaded in step 1
splitter = RecursiveCharacterTextSplitter(chunk_size=200,chunk_overlap=50)
splitted_docs = splitter.split_documents(docs)
len(splitted_docs)

125

In [75]:
# Step 3 : Convert the Splitted Docs Vector Embedding (using embedding model) and store them in Vector DB
embeddings = NVIDIAEmbeddings(model="nvidia/nemotron-3-embed-1b")
# embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")
vector_store = InMemoryVectorStore.from_documents(
    documents=splitted_docs,
    embedding=embedder
)

In [76]:
# step 4 Build an Agent ( Agent need : tools , llm, prompt)

# Creating Tool
@tool
def retriever_tool(query:str):
    """
        This tool can help you to retrieve the relevant data from the PDF Document, and this documents have details about medical Reprots 
    """
    docs = vector_store.similarity_search(query=query,k=4)
    context = ""
    print("Tool Called : ", query)
    for doc in docs:
        context += doc.page_content + "\n\n"
    
    return context

# Creating LLM
llm = ChatGoogleGenerativeAI(model='gemini-3.5-flash-lite')


# System Prompt
system_prompt = """
    You are a helpful assistant that answer the question using retrieved contex.
    Always use the 'retriever_tool' tool for questions requiring external knoledge.
"""


In [77]:
agent = create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt=system_prompt
)

In [ ]:
query = "What is the name of the Patient, and tell me about Anti CCP antibodies  of the patient"

In [78]:
result = agent.invoke({"messages":[{"role":"user","content":query}]})

Tool Called :  Patient name Anti CCP antibodies
Tool Called :  ANTI CCP value result
Tool Called :  ANTI CCP SERUM results value


In [79]:
print(result["messages"][-1].content[0]['text'])

Based on the retrieved medical report:

* **Patient Name:** Mr. Rachel V. John
* **Anti-CCP Antibodies (Cyclic Citrullinated Peptide, Serum):** 
  * The result is listed as **`<17.00`** (with a biological reference interval of `<8.0` U/mL, and the assay having a sensitivity of 70.6% and specificity of 98.2%). 
  * **Clinical Context:** Anti-CCP antibodies are useful for evaluating patients suspected of having rheumatoid arthritis. Their clinical utility for diagnosing rheumatoid arthritis is far greater than that of the rheumatoid factor. False-positive results are uncommon, and up to 30% of patients with seronegative rheumatoid arthritis also show Anti-CCP antibodies.
